In [1]:
import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit, RandomizedSearchCV, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = "/content/"
DATA_CSV    = os.path.join(BASE_DIR, "processed_data.csv")
ENCODER_PKL = os.path.join(BASE_DIR, "station_encoder.pkl")
PLOT_DIR    = os.path.join(BASE_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

FEATURE_COLS = [
    "station_encoded", "destination_encoded", "arrival_delay",
    "planned_arr_mins", "day_of_week", "week_of_year", "is_peak_hour",
]
TARGET_COL = "target_delay_at_destination"

PALETTE = ["#20808D", "#A84B2F", "#1B474D", "#944454", "#FFC553", "#848456"]

plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#E0DDD8",
    "grid.linewidth": 0.6,
})

# ── 1. Load data ───────────────────────────────────────────────────────────────
print("=== Loading dataset ===")
df = pd.read_csv(DATA_CSV)
with open(ENCODER_PKL, "rb") as f:
    encoder = pickle.load(f)

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values
groups = df["journey_id"].values
print(f"  {len(df):,} rows | {df['journey_id'].nunique():,} journeys")

# ── 2. Train/test split (by journey — same as before) ─────────────────────────
print("\n=== Train/Test Split ===")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test   = X[train_idx], X[test_idx]
y_train, y_test   = y[train_idx], y[test_idx]
g_train           = groups[train_idx]
print(f"  Train: {len(X_train):,}  |  Test: {len(X_test):,}")

# Cross-val splitter (group-aware, used inside RandomizedSearchCV)
cv = GroupKFold(n_splits=3)

# ── 3. Baseline models (default params from model_comparison.py) ───────────────
print("\n=== Baseline models ===")

baseline_models = {
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  Ridge(alpha=1.0)),
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42 # Reduced n_estimators
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.05, # Reduced n_estimators
        random_state=42, n_jobs=-1, verbosity=0
    ),
    "LightGBM": lgb.LGBMRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.05, # Reduced n_estimators
        random_state=42, n_jobs=-1, verbose=-1
    ),
}

def evaluate(model, X_tr, y_tr, X_te, y_te, fit=True):
    if fit:
        model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    return dict(
        RMSE    = float(np.sqrt(mean_squared_error(y_te, y_pred))),
        MAE     = float(mean_absolute_error(y_te, y_pred)),
        R2      = float(r2_score(y_te, y_pred)),
        Within2 = float(np.mean(np.abs(y_pred - y_te) <= 2)  * 100),
        Within5 = float(np.mean(np.abs(y_pred - y_te) <= 5)  * 100),
        Within10= float(np.mean(np.abs(y_pred - y_te) <= 10) * 100),
        y_pred  = y_pred,
    )

baseline_results = {}
for name, model in baseline_models.items():
    print(f"  {name}...", end=" ", flush=True)
    res = evaluate(model, X_train, y_train, X_test, y_test)
    baseline_results[name] = res
    print(f"MAE={res['MAE']:.3f}  RMSE={res['RMSE']:.3f}  R²={res['R2']:.4f}")

# ── 4. Hyperparameter search spaces ───────────────────────────────────────────
print("\n=== Hyperparameter Tuning (RandomizedSearchCV) ===")

search_configs = {
    "Ridge": {
        "estimator": Pipeline([("scaler", StandardScaler()), ("model", Ridge())]),
        "param_dist": {
            "model__alpha": [0.1, 1.0, 10.0, 100.0],
        },
        "n_iter": 4,
        "fit_params": {},
    },
    "Random Forest": {
        "estimator": RandomForestRegressor(random_state=42, n_jobs=1), # Changed n_jobs to 1
        "param_dist": {
            "n_estimators": [50, 100], # Reduced from [100, 200]
            "max_depth":    [5, 10],   # Removed None from here
            "max_features": ["sqrt", 0.5],
        },
        "n_iter": 3, # Reduced n_iter to 3
        "fit_params": {},
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingRegressor(random_state=42),
        "param_dist": {
            "n_estimators":  [50, 100], # Reduced from [100, 200]
            "max_depth":     [3, 4],    # Reduced from [3, 5]
            "learning_rate": [0.05, 0.1],
            "subsample":     [0.8, 1.0],
        },
        "n_iter": 8,
        "fit_params": {},
    },
    "XGBoost": {
        "estimator": xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0),
        "param_dist": {
            "n_estimators":     [50, 100], # Reduced from [100, 200]
            "max_depth":        [4, 5],   # Reduced from [4, 6]
            "learning_rate":    [0.05, 0.1],
            "subsample":        [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
        },
        "n_iter": 4, # Reduced from 8
        "fit_params": {},
    },
    "LightGBM": {
        "estimator": lgb.LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
        "param_dist": {
            "n_estimators":  [50, 100], # Reduced from [100, 200]
            "max_depth":     [5, 6],    # Adjusted from [6, -1]
            "learning_rate": [0.05, 0.1],
            "num_leaves":    [31, 63],
            "subsample":     [0.8, 1.0],
        },
        "n_iter": 4, # Reduced from 8
        "fit_params": {},
    },
}

tuned_models  = {}
tuned_results = {}
best_params   = {}

for name, cfg in search_configs.items():
    print(f"\n  Tuning {name} ({cfg['n_iter']} iterations × 3-fold CV)...")

    # Determine n_jobs for RandomizedSearchCV based on the model name
    # Set n_jobs to 1 for Random Forest to prevent OOM errors, keep -1 for others
    rs_n_jobs = 1 if name == "Random Forest" else -1

    search = RandomizedSearchCV(
        estimator          = cfg["estimator"],
        param_distributions= cfg["param_dist"],
        n_iter             = cfg["n_iter"],
        cv                 = cv,
        scoring            = "neg_mean_absolute_error",
        random_state       = 42,
        n_jobs             = rs_n_jobs, # Modified n_jobs here
        verbose            = 0,
        refit              = True,
    )
    search.fit(X_train, y_train, groups=g_train)
    best_params[name] = search.best_params_
    print(f"    Best params: {search.best_params_}")

    tuned_model = search.best_estimator_
    tuned_models[name] = tuned_model
    res = evaluate(tuned_model, X_train, y_train, X_test, y_test, fit=False)
    tuned_results[name] = res
    improvement = baseline_results[name]["MAE"] - res["MAE"]
    print(f"    MAE={res['MAE']:.3f}  RMSE={res['RMSE']:.3f}  R²={res['R2']:.4f}  "
          f"(ΔMAE={improvement:+.3f})")

# Save best params
with open(os.path.join(BASE_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2, default=str)
print(f"\n  Best params saved to {BASE_DIR}/best_params.json")

# ── 5. Plots ───────────────────────────────────────────────────────────────────
model_names = list(baseline_results.keys())
colors_base = PALETTE[:len(model_names)]
colors_tune = [c + "AA" for c in colors_base]   # slightly transparent version — handled via alpha

# ── Plot A: Before vs After — MAE & RMSE side by side ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Hyperparameter Tuning — Before vs After\n(Waterloo ↔ Weymouth, 2022–2025)",
             fontsize=14, fontweight="bold", y=1.02)

x     = np.arange(len(model_names))
width = 0.35

for ax, metric, title, ylabel in [
    (axes[0], "MAE",  "Mean Absolute Error (MAE)",      "minutes"),
    (axes[1], "RMSE", "Root Mean Squared Error (RMSE)", "minutes"),
]:
    base_vals  = [baseline_results[n][metric] for n in model_names]
    tuned_vals = [tuned_results[n][metric]    for n in model_names]

    b1 = ax.bar(x - width/2, base_vals,  width, label="Baseline",
                color=PALETTE[0], alpha=0.7, edgecolor="white")
    b2 = ax.bar(x + width/2, tuned_vals, width, label="Tuned",
                color=PALETTE[1], alpha=0.9, edgecolor="white")

    for bar, v in zip(b1, base_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8, color="#28251D")
    for bar, v in zip(b2, tuned_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.2f}", ha="center", va="bottom", fontsize=8, color="#28251D")

    ax.set_xticks(x)
    ax.set_xticklabels(model_names, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(base_vals + tuned_vals) * 1.18)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/07_tuning_before_after.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n  ✓ saved 07_tuning_before_after.png")

# ── Plot B: R² before vs after ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
base_r2  = [baseline_results[n]["R2"] for n in model_names]
tuned_r2 = [tuned_results[n]["R2"]   for n in model_names]

b1 = ax.bar(x - width/2, base_r2,  width, label="Baseline",
            color=PALETTE[0], alpha=0.7, edgecolor="white")
b2 = ax.bar(x + width/2, tuned_r2, width, label="Tuned",
            color=PALETTE[1], alpha=0.9, edgecolor="white")

all_r2 = base_r2 + tuned_r2
y_range = max(abs(min(all_r2)), abs(max(all_r2)))
for bar, v in zip(list(b1) + list(b2), base_r2 + tuned_r2):
    xp = bar.get_x() + bar.get_width()/2
    yp = v - y_range * 0.04 if v < 0 else v + y_range * 0.01
    va = "top" if v < 0 else "bottom"
    ax.text(xp, yp, f"{v:.3f}", ha="center", va=va, fontsize=8, color="#28251D")

ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel("R²")
ax.set_title("R² Score — Baseline vs Tuned")
ax.legend(fontsize=9)
ax.axhline(0, color="#28251D", lw=0.8, linestyle="--")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/08_tuning_r2.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 08_tuning_r2.png")

# ── Plot C: MAE improvement (delta) ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
deltas = [baseline_results[n]["MAE"] - tuned_results[n]["MAE"] for n in model_names]
bar_colors = [PALETTE[3] if d < 0 else PALETTE[2] for d in deltas]
bars = ax.bar(model_names, deltas, color=bar_colors, edgecolor="white", width=0.5)
ax.axhline(0, color="#28251D", lw=1.0, linestyle="--")
for bar, v in zip(bars, deltas):
    yp = v + 0.001 if v >= 0 else v - 0.001
    va = "bottom" if v >= 0 else "top"
    ax.text(bar.get_x() + bar.get_width()/2, yp,
            f"{v:+.3f} min", ha="center", va=va, fontsize=9, fontweight="bold")
ax.set_ylabel("MAE reduction (minutes)")
ax.set_title("MAE Improvement from Tuning\n(positive = better after tuning)")
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/09_mae_improvement.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 09_mae_improvement.png")

# ── Plot D: Within-N-minutes accuracy after tuning ────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
x2    = np.arange(len(model_names))
width = 0.25

for i, (tol, col) in enumerate([(2, PALETTE[0]), (5, PALETTE[1]), (10, PALETTE[2])]):
    vals = [tuned_results[n][f"Within{tol}"] for n in model_names]
    bars = ax.bar(x2 + (i - 1) * width, vals, width,
                  label=f"Within {tol} min", color=col,
                  edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.4,
                f"{v:.0f}%", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x2)
ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel("% of Predictions")
ax.set_title("Within-N-Minutes Accuracy — Tuned Models")
ax.legend(loc="lower right", fontsize=9)
ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/10_tuned_within_n.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 10_tuned_within_n.png")

# ── Plot E: Actual vs Predicted — Tuned models ────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle("Actual vs Predicted — Tuned Models", fontsize=14, fontweight="bold")
axes = axes.flatten()
sample = np.random.RandomState(42).choice(len(y_test),
         size=min(5000, len(y_test)), replace=False)
lim_lo, lim_hi = -15, 45

for ax, (name, col) in zip(axes, zip(model_names, PALETTE)):
    y_pred = tuned_results[name]["y_pred"]
    ax.scatter(y_test[sample], y_pred[sample],
               alpha=0.25, s=4, color=col, rasterized=True)
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", lw=1.2, label="Perfect")
    ax.set_xlim(lim_lo, lim_hi)
    ax.set_ylim(lim_lo, lim_hi)
    ax.set_xlabel("Actual (min)")
    ax.set_ylabel("Predicted (min)")
    r2  = tuned_results[name]["R2"]
    mae = tuned_results[name]["MAE"]
    ax.set_title(f"{name}\nR²={r2:.3f}  MAE={mae:.2f} min")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/11_tuned_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 11_tuned_actual_vs_predicted.png")

# ── Plot F: Summary heatmap — tuned models ────────────────────────────────────
metrics_display = ["RMSE\n(min)", "MAE\n(min)", "R²",
                   "Within\n2 min (%)", "Within\n5 min (%)", "Within\n10 min (%)"]
metric_keys = ["RMSE", "MAE", "R2", "Within2", "Within5", "Within10"]

raw_values = np.array([
    [tuned_results[n][k] for k in metric_keys]
    for n in model_names
])

norm = raw_values.copy().astype(float)
for col_i, key in enumerate(metric_keys):
    lo, hi = norm[:, col_i].min(), norm[:, col_i].max()
    if hi == lo:
        norm[:, col_i] = 0.5
        continue
    if key in ("RMSE", "MAE"):
        norm[:, col_i] = 1 - (norm[:, col_i] - lo) / (hi - lo)
    else:
        norm[:, col_i] = (norm[:, col_i] - lo) / (hi - lo)

fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(norm, cmap=plt.get_cmap("YlGn"), aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(6))
ax.set_xticklabels(metrics_display, fontsize=10)
ax.set_yticks(range(len(model_names)))
ax.set_yticklabels(model_names, fontsize=10)
ax.set_title("Tuned Model Performance Heatmap\n(green = better, per-column normalised)",
             fontsize=13, fontweight="bold")

for i in range(len(model_names)):
    for j, key in enumerate(metric_keys):
        raw = raw_values[i, j]
        text = f"{raw:.3f}" if key == "R2" else f"{raw:.1f}"
        txt_col = "black" if norm[i, j] > 0.45 else "white"
        ax.text(j, i, text, ha="center", va="center",
                fontsize=9, color=txt_col, fontweight="bold")

plt.colorbar(im, ax=ax, label="Relative performance (1=best)", shrink=0.8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/12_tuned_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ saved 12_tuned_heatmap.png")

# ── Final summary table ────────────────────────────────────────────────────────
print("\n" + "="*70)
print(f"{'Model':<20} {'Base MAE':>9} {'Tuned MAE':>10} {'ΔMAE':>8} {'Tuned R²':>10}")
print("-"*70)
for name in model_names:
    b = baseline_results[name]["MAE"]
    t = tuned_results[name]["MAE"]
    r = tuned_results[name]["R2"]
    print(f"{name:<20} {b:>9.3f} {t:>10.3f} {t-b:>+8.3f} {r:>10.4f}")
print("="*70)
print(f"\nAll plots saved to: {PLOT_DIR}")

=== Loading dataset ===
  1,485,507 rows | 19,945 journeys

=== Train/Test Split ===
  Train: 1,187,531  |  Test: 297,976

=== Baseline models ===
  Ridge... MAE=3.642  RMSE=7.093  R²=0.3466
  Random Forest... MAE=3.338  RMSE=6.752  R²=0.4079
  Gradient Boosting... MAE=3.379  RMSE=6.748  R²=0.4086
  XGBoost... MAE=3.245  RMSE=6.578  R²=0.4380
  LightGBM... MAE=3.259  RMSE=6.596  R²=0.4350

=== Hyperparameter Tuning (RandomizedSearchCV) ===

  Tuning Ridge (4 iterations × 3-fold CV)...
    Best params: {'model__alpha': 0.1}
    MAE=3.642  RMSE=7.093  R²=0.3466  (ΔMAE=+0.000)

  Tuning Random Forest (3 iterations × 3-fold CV)...
    Best params: {'n_estimators': 100, 'max_features': 'sqrt', 'max_depth': 10}
    MAE=3.441  RMSE=6.754  R²=0.4076  (ΔMAE=-0.103)

  Tuning Gradient Boosting (8 iterations × 3-fold CV)...
    Best params: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.1}
    MAE=3.297  RMSE=6.667  R²=0.4228  (ΔMAE=+0.083)

  Tuning XGBoost (8 iterati